In [ ]:
import os
import subprocess
import time
import requests
import signal
import sys

def screen_exists(screen_name):
    """Check if a screen session exists."""
    try:
        result = subprocess.run(
            ['screen', '-list'],
            capture_output=True,
            text=True,
            check=True
        )
        return screen_name in result.stdout
    except subprocess.CalledProcessError:
        return False

def start_or_resume_screen(screen_name, command=None, env_vars=None):
    """Start a new screen session or resume an existing one."""
    if screen_exists(screen_name):
        print(f"Resuming existing screen {screen_name}")
        if command:
            # Attach to screen, run command, then detach
            env_cmd = ""
            if env_vars:
                env_cmd = " ".join([f"export {k}={v};" for k, v in env_vars.items()])

            full_cmd = f"screen -S {screen_name} -X stuff '{env_cmd} {command}\n'"
            subprocess.run(full_cmd, shell=True, check=True)
    else:
        print(f"Creating new screen {screen_name}")
        # Create a new detached screen session
        env_string = ""
        if env_vars:
            env_string = " ".join([f"{k}={v}" for k, v in env_vars.items()])

        if command:
            screen_cmd = f"{env_string} screen -dmS {screen_name} bash -c '{command}'"
        else:
            screen_cmd = f"{env_string} screen -dmS {screen_name}"

        subprocess.run(screen_cmd, shell=True, check=True)

def check_server_health(url="http://localhost:8000/health/", max_attempts=30, delay=10):
    """Check if the VLLM server is healthy by polling the /health/ endpoint."""
    print(f"Waiting for VLLM server to be ready at {url}")
    for attempt in range(max_attempts):
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                print("VLLM server is ready!")
                return True
        except requests.RequestException:
            pass

        print(f"Server not ready, checking again in {delay} seconds... (attempt {attempt+1}/{max_attempts})")
        time.sleep(delay)

    print("Server health check timed out!")
    return False

def get_available_gpus():
    """Get a list of available CUDA device IDs."""
    try:
        # Try using nvidia-smi to get GPU count
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=index', '--format=csv,noheader'],
            capture_output=True,
            text=True,
            check=True
        )
        gpu_indices = [line